# Giai đoạn 1: Tiền xử lý dữ liệu MIMIC-III/IV PPG
Trích xuất đặc trưng HRV (13 đặc trưng) từ tập dữ liệu MIMIC nguyên thủy.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt, welch
from scipy.interpolate import interp1d
import os

### 1. Định nghĩa bộ lọc nhiễu (Bandpass Filter)
Dữ liệu MIMIC thô rất nhiễu, cần được lọc để loại bỏ nhiễu đường nền và nhiễu tần số cao trước khi tìm đỉnh.

In [2]:
def butter_bandpass_filter(data, lowcut=0.5, highcut=8.0, fs=125.0, order=3):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

### 2. Các hàm trích xuất đặc trưng (Time & Frequency Domain)

In [3]:
def calculate_time_domain(rr_intervals):
    if len(rr_intervals) < 2:
        return {k: np.nan for k in ['Mean_NN', 'SDNN', 'RMSSD', 'pNN50', 'NN50', 'CV']}
    
    mean_nn = np.mean(rr_intervals)
    sdnn = np.std(rr_intervals, ddof=1)
    
    diff_nn = np.diff(rr_intervals)
    rmssd = np.sqrt(np.mean(diff_nn**2))
    
    nn50 = np.sum(np.abs(diff_nn) > 50)
    pnn50 = (nn50 / len(diff_nn)) * 100 if len(diff_nn) > 0 else 0
    
    cv = (sdnn / mean_nn) * 100 if mean_nn > 0 else np.nan
    
    return {
        'Mean_NN': mean_nn,
        'SDNN': sdnn,
        'RMSSD': rmssd,
        'pNN50': pnn50,
        'NN50': nn50,
        'CV': cv
    }

def calculate_frequency_domain(rr_intervals, fs_interp=4.0):
    nan_result = {k: np.nan for k in ['LF', 'HF', 'LF_HF_Ratio', 'LF_norm', 'HF_norm', 'Total_Power']}
    if len(rr_intervals) < 10:
        return nan_result
    
    t = np.cumsum(rr_intervals) / 1000.0
    t -= t[0]
    
    rr_sec = rr_intervals / 1000.0
    
    try:
        f_interp = interp1d(t, rr_sec, kind='cubic')
        t_interp = np.arange(t[0], t[-1], 1.0 / fs_interp)
        rr_interp = f_interp(t_interp)
    except Exception:
        return nan_result
        
    nperseg = min(256, len(rr_interp))
    if nperseg < 2:
        return nan_result
        
    f, pxx = welch(rr_interp, fs=fs_interp, nperseg=nperseg)
    
    lf_band = (f >= 0.04) & (f <= 0.15)
    hf_band = (f >= 0.15) & (f <= 0.4)
    
    df = f[1] - f[0]
    lf = np.sum(pxx[lf_band]) * df
    hf = np.sum(pxx[hf_band]) * df
    
    total_power = lf + hf
    lf_hf_ratio = lf / hf if hf > 0 else np.nan
    
    lf_norm = (lf / total_power) * 100 if total_power > 0 else np.nan
    hf_norm = (hf / total_power) * 100 if total_power > 0 else np.nan
    
    return {
        'LF': lf,
        'HF': hf,
        'LF_HF_Ratio': lf_hf_ratio,
        'LF_norm': lf_norm,
        'HF_norm': hf_norm,
        'Total_Power': total_power
    }

### 3. Hàm trích xuất đặc trưng từ cửa sổ 30 giây

In [4]:
def extract_features_mimic(time_array, ppg_array, fs=125.0):
    # 1. Lọc nhiễu
    clean_ppg = butter_bandpass_filter(ppg_array, fs=fs)
    
    # 2. Tìm đỉnh (Peak Detection)
    min_distance = int(0.25 * fs) # Khoảng cách tối thiểu giữa 2 nhịp tim (~240 BPM)
    peaks, _ = find_peaks(clean_ppg, distance=min_distance, prominence=np.std(clean_ppg)*0.5)
    
    if len(peaks) < 10:
        return None
        
    # 3. Tính R-R Intervals (ms)
    peak_times = time_array[peaks]
    rr_intervals = np.diff(peak_times) * 1000.0
    
    # 4. BỘ LỌC SINH LÝ: Bỏ các RR vô lý (<300ms hoặc >2000ms)
    valid_rr = rr_intervals[(rr_intervals >= 300) & (rr_intervals <= 2000)]
    
    if len(valid_rr) < 10:
        return None
        
    # 5. Tính 13 đặc trưng
    features = {'HR_mean': 60000.0 / np.mean(valid_rr)}
    features.update(calculate_time_domain(valid_rr))
    features.update(calculate_frequency_domain(valid_rr))
    
    # Xử lý NaN
    for k, v in features.items():
        if np.isnan(v):
            features[k] = 0.0
            
    return features

### 4. Đọc dữ liệu và trượt cửa sổ

In [5]:
# Đọc dữ liệu raw
df = pd.read_csv('../../data/raw/mimic_ppg_af_dataset.csv')
print(f"Tổng số điểm dữ liệu: {len(df)}")

FS = 125.0
WINDOW_SIZE_SEC = 30
window_size_samples = int(WINDOW_SIZE_SEC * FS)

features_list = []
total_samples = len(df)

print("Bắt đầu trích xuất...")
for start_idx in range(0, total_samples, window_size_samples):
    end_idx = start_idx + window_size_samples
    if end_idx > total_samples:
        break
        
    window = df.iloc[start_idx:end_idx]
    
    # Lấy nhãn của cửa sổ (lấy nhãn phổ biến nhất hoặc nhãn ở cuối cửa sổ)
    status_vals = window['status'].values
    label = 1 if np.mean(status_vals) > 0.5 else 0
    
    time_arr = window['time'].values
    ppg_arr = window['ppg'].values
    
    feats = extract_features_mimic(time_arr, ppg_arr, fs=FS)
    
    if feats is not None:
        feats['status'] = label
        features_list.append(feats)
        
    if (start_idx // window_size_samples) % 500 == 0:
        print(f"Đã xử lý {start_idx // window_size_samples} cửa sổ...")

out_df = pd.DataFrame(features_list)
print(f"Số mẫu (cửa sổ 30s) hợp lệ thu được: {len(out_df)}")

# Lưu kết quả
os.makedirs('../../data/features', exist_ok=True)
out_df.to_csv('../../data/features/mimic_train_features.csv', index=False)
print("Đã lưu vào data/features/mimic_train_features.csv")

Tổng số điểm dữ liệu: 5250035
Bắt đầu trích xuất...
Đã xử lý 0 cửa sổ...
Đã xử lý 500 cửa sổ...
Đã xử lý 1000 cửa sổ...
Số mẫu (cửa sổ 30s) hợp lệ thu được: 1342
Đã lưu vào data/features/mimic_train_features.csv
